# Business Entity Resolution Pipeline — Phase 5
## Scoring Model, Threshold Optimization & Leaderboard Inference
**Team:** `GenX H4CK3RS!`  
**Competition:** Amazon ML Challenge 2026 — Business Entity Resolution  
**Evaluation Metric:** Exact Macro-averaged $F_{0.5}$ with Strict Singleton Rules  

---

### Pipeline Architecture:
1. **Colab T4 GPU Acceleration & Environment Check**: Configure LightGBM to utilize GPU acceleration (`device="gpu"` / `device_type="cuda"`) with safe multi-threaded CPU fallback.
2. **Exact Macro $F_{0.5}$ Evaluation Metric**: Vectorized metric harness strictly enforcing competition singleton scoring (empty predictions on true singletons $= 1.0$, false merges $= 0.0$).
3. **Feature Matrix Ingestion**: Ingestion of the 33-dimensional pairwise feature space engineered during Phase 4.
4. **LightGBM Binary Classifier Training**: Calibrated `scale_pos_weight` to address the ~1:1600 candidate class imbalance.
5. **Precision-Heavy Threshold Grid Search**: Grid search across decision thresholds ($0.40 - 0.90$) penalizing false positives twice as heavily as false negatives.
6. **Leaderboard TSV Generation**: Full inference generating `output/matching_results.tsv` and `output/candidate_pairs.tsv` covering all 1.73M test records.
7. **Submission Packaging & Validation**: Automatic validation using official `validate_submission.py` and packaging into `GenX_H4CK3RS!_submission.zip`.


In [ ]:
# ==============================================================================
# Cell 0: Google Colab Setup (Run this FIRST if running in Google Colab)
# ==============================================================================
import os, sys
from pathlib import Path

# If on Colab server and repo is not yet cloned, clone and cd into it
if Path("/content").exists() and not Path("code/business_entity_resolution").exists():
    print("[*] Detected Google Colab server. Cloning repository...")
    !git clone https://github.com/AnushaNatesan/Amazon-ML-Challenge.git
    %cd Amazon-ML-Challenge
    !pip install -q rapidfuzz lightgbm
    print("[+] Colab environment setup complete!")
else:
    print("[*] Repository already present in working directory.")


In [8]:
# ==============================================================================
# Cell 1: Environment Setup, Sys Path Configuration & GPU Acceleration Detection
# ==============================================================================
import os
import sys
import time
from pathlib import Path
import numpy as np
import pandas as pd

# 1. Resolve Workspace Root (Supports Google Colab, Kaggle, and Local environments)
current_dir = Path.cwd()
if (current_dir / "code" / "business_entity_resolution").exists():
    workspace_root = current_dir
elif (current_dir / "Amazon-ML-Challenge" / "code" / "business_entity_resolution").exists():
    workspace_root = current_dir / "Amazon-ML-Challenge"
    os.chdir(workspace_root)
else:
    workspace_root = current_dir

code_dir = workspace_root / "code" / "business_entity_resolution"
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

print(f"[*] Workspace Root: {workspace_root}")
print(f"[*] Python Version: {sys.version.split()[0]}")

# 2. Check for GPU (Google Colab T4 or CUDA backend)
gpu_available = False
device_param = "cpu"

try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        print(f"[+] CUDA GPU Detected: {gpu_name} (Colab T4 Backend Ready)")
        gpu_available = True
        device_param = "gpu"
    else:
        print("[-] No CUDA GPU detected via PyTorch. Using multi-threaded CPU mode.")
except ImportError:
    import shutil
    if shutil.which("nvidia-smi"):
        print("[+] nvidia-smi detected. Configuring GPU acceleration.")
        gpu_available = True
        device_param = "gpu"
    else:
        print("[-] GPU tools not detected. Using multi-threaded CPU mode.")

# Ensure output directory exists
os.makedirs(workspace_root / "output", exist_ok=True)
print(f"[+] Target Output Directory: {workspace_root / 'output'}")


[*] Workspace Root: /content
[*] Python Version: 3.12.13
[+] CUDA GPU Detected: Tesla T4 (Colab T4 Backend Ready)
[+] Target Output Directory: /content/output


In [9]:
# ==============================================================================
# Cell 2: Exact Macro-Averaged F_0.5 Evaluation Harness (With Strict Singleton Rule)
# ==============================================================================
from typing import Dict, List, Set, Union

def compute_entity_f05(
    pred_matches: Union[Set[str], List[str], str],
    true_matches: Union[Set[str], List[str], str],
) -> float:
    """
    Computes F_0.5 for a single Source 1 entity:
    F_0.5 = (1.25 * Precision * Recall) / (0.25 * Precision + Recall)
    
    Singleton Rules:
    - True singleton + predicted empty = 1.0 (correctly identified singleton)
    - True singleton + predicted matches = 0.0 (penalizing false merge)
    - Non-singleton + predicted empty = 0.0
    """
    if isinstance(pred_matches, str):
        p_set = {x.strip() for x in pred_matches.split(",") if x.strip()}
    else:
        p_set = {x for x in pred_matches if str(x).strip()}

    if isinstance(true_matches, str):
        t_set = {x.strip() for x in true_matches.split(",") if x.strip()}
    else:
        t_set = {x for x in true_matches if str(x).strip()}

    # 1. Singleton case (no true matches)
    if len(t_set) == 0:
        return 1.0 if len(p_set) == 0 else 0.0

    # 2. Non-singleton case
    if len(p_set) == 0:
        return 0.0

    tp = len(p_set & t_set)
    if tp == 0:
        return 0.0

    precision = tp / len(p_set)
    recall = tp / len(t_set)

    denominator = (0.25 * precision) + recall
    if denominator == 0.0:
        return 0.0

    return (1.25 * precision * recall) / denominator


def calculate_macro_f05(
    y_true_dict: Dict[str, Set[str]],
    y_pred_dict: Dict[str, Set[str]],
) -> float:
    """Calculates macro-averaged F_0.5 score across all Source 1 entities in evaluation set."""
    scores = []
    for s1_id, true_set in y_true_dict.items():
        pred_set = y_pred_dict.get(s1_id, set())
        score = compute_entity_f05(pred_set, true_set)
        scores.append(score)
    return float(np.mean(scores)) if scores else 0.0

# Verify against official edge cases
assert compute_entity_f05([], []) == 1.0, "Singleton empty must score 1.0"
assert compute_entity_f05(["S2-123"], []) == 0.0, "Singleton false match must score 0.0"
assert compute_entity_f05([], ["S2-123"]) == 0.0, "Non-singleton empty must score 0.0"
assert compute_entity_f05(["S2-123"], ["S2-123"]) == 1.0, "Perfect match must score 1.0"
print("✓ Exact Macro F_0.5 evaluation harness compiled and verified.")


✓ Exact Macro F_0.5 evaluation harness compiled and verified.


In [10]:
# ==============================================================================
# Cell 3: Feature Matrix Ingestion & Train/Validation Stratification
# ==============================================================================
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# Ensure code_dir and src are in sys.path regardless of execution order
candidates = [
    Path.cwd() / "code" / "business_entity_resolution",
    Path("/content/Amazon-ML-Challenge/code/business_entity_resolution"),
    Path("/content/code/business_entity_resolution"),
]
for p in candidates:
    if p.exists():
        if str(p) not in sys.path:
            sys.path.insert(0, str(p))
        if str(p / "src") not in sys.path:
            sys.path.insert(0, str(p / "src"))
        break

try:
    from src.features import FEATURE_NAMES
except ModuleNotFoundError:
    try:
        from features import FEATURE_NAMES
    except ModuleNotFoundError:
        # Last resort: find features.py anywhere in filesystem
        feat_files = list(Path.cwd().glob("**/features.py"))
        if not feat_files and Path("/content").exists():
            feat_files = list(Path("/content").glob("**/features.py"))
        if feat_files:
            src_dir = feat_files[0].parent
            sys.path.insert(0, str(src_dir.parent))
            sys.path.insert(0, str(src_dir))
            from src.features import FEATURE_NAMES
        else:
            raise

import lightgbm as lgb
from sklearn.model_selection import train_test_split

# Resolve workspace_root
if "workspace_root" not in globals():
    workspace_root = Path.cwd()
    if (workspace_root / "Amazon-ML-Challenge").exists():
        workspace_root = workspace_root / "Amazon-ML-Challenge"

feature_file = workspace_root / "output" / "train_features.csv"
if not feature_file.exists() and Path("/content/Amazon-ML-Challenge/output/train_features.csv").exists():
    feature_file = Path("/content/Amazon-ML-Challenge/output/train_features.csv")

if feature_file.exists():
    print(f"[*] Ingesting pre-computed feature matrix from: {feature_file}")
    df_feat = pd.read_csv(feature_file)
else:
    print(f"[!] {feature_file} not found. Running fast feature extraction pipeline...")
    try:
        from run_feature_extraction import run_feature_extraction_pipeline
    except ModuleNotFoundError:
        from code.business_entity_resolution.run_feature_extraction import run_feature_extraction_pipeline
    run_feature_extraction_pipeline(sample_size=300, output_path=str(feature_file))
    df_feat = pd.read_csv(feature_file)

print(f"[+] Ingested {len(df_feat):,} candidate pair records with {len(FEATURE_NAMES)} features.")

# Extract feature matrix X and binary target y
X = df_feat[FEATURE_NAMES].values
y = df_feat["is_match"].values
pair_s1 = df_feat["source1_entity_id"].values
pair_cand = df_feat["candidate_entity_id"].values

num_pos = int(np.sum(y == 1))
num_neg = int(np.sum(y == 0))
imbalance_ratio = num_neg / max(1, num_pos)
print(f"  - Positive Match Pairs:     {num_pos:,} ({num_pos / len(y) * 100:.2f}%)")
print(f"  - Negative Distractor Pairs: {num_neg:,} ({num_neg / len(y) * 100:.2f}%)")
print(f"  - Candidate Imbalance Ratio: {imbalance_ratio:.1f} : 1")

# Group-stratified train/validation split by Source 1 entity to prevent data leakage
unique_s1 = np.unique(pair_s1)
s1_train, s1_val = train_test_split(unique_s1, test_size=0.25, random_state=42)
val_mask = np.isin(pair_s1, s1_val)
train_mask = ~val_mask

X_train, y_train = X[train_mask], y[train_mask]
X_val, y_val = X[val_mask], y[val_mask]
s1_val_pairs = pair_s1[val_mask]
cand_val_pairs = pair_cand[val_mask]

print(f"[+] Train split: {len(X_train):,} pairs ({len(s1_train):,} S1 entities)")
print(f"[+] Val split:   {len(X_val):,} pairs ({len(s1_val):,} S1 entities)")


In [4]:
# ==============================================================================
# Cell 4: Train LightGBM Classifier with Class Imbalance & GPU Acceleration
# ==============================================================================
# Calibrate scale_pos_weight for candidate imbalance
scale_weight = min(25.0, max(5.0, imbalance_ratio * 0.5))

lgb_params = {
    "objective": "binary",
    "metric": "binary_logloss",
    "boosting_type": "gbdt",
    "n_estimators": 300,
    "learning_rate": 0.05,
    "num_leaves": 31,
    "max_depth": 6,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "scale_pos_weight": scale_weight,
    "random_state": 42,
    "verbose": -1,
}

# Configure GPU if available (Colab T4 runtime)
if gpu_available:
    try:
        lgb_params["device"] = "gpu"
        print("[*] Testing LightGBM with GPU parameter: device='gpu'...")
        test_model = lgb.LGBMClassifier(**lgb_params)
        test_model.fit(X_train[:100], y_train[:100])
        print("[+] LightGBM successfully initialized on GPU runtime!")
    except Exception as e:
        print(f"[-] GPU initialization fallback: {e}")
        lgb_params.pop("device", None)
        lgb_params["n_jobs"] = -1
else:
    lgb_params["n_jobs"] = -1

print(f"[*] Training LightGBM classifier with scale_pos_weight={scale_weight:.1f}...")
model = lgb.LGBMClassifier(**lgb_params)
model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(stopping_rounds=40, verbose=False)],
)

# Feature Importance Analysis
importances = model.feature_importances_
top_idx = np.argsort(importances)[::-1][:10]
print("\n[+] Top 10 Most Predictive Features:")
for i, idx in enumerate(top_idx, 1):
    print(f"  {i:2d}. {FEATURE_NAMES[idx]:<30}: {int(importances[idx]):5d}")


In [7]:
# ==============================================================================
# Cell 5: Precision-Heavy Macro F_0.5 Threshold Optimization
# ==============================================================================
from collections import defaultdict

# Predict probabilities on validation candidate pairs
val_probs = model.predict_proba(X_val)[:, 1]

# Build validation ground truth dictionary
val_gt_dict = {s1: set() for s1 in s1_val}
for s1, cand, label in zip(s1_val_pairs, cand_val_pairs, y_val):
    if label == 1:
        val_gt_dict[s1].add(cand)

# Grid search across decision thresholds from 0.40 to 0.90
thresholds = np.arange(0.40, 0.92, 0.02)
best_threshold = 0.30
best_f05 = -1.0
threshold_results = []

print("[*] Optimizing decision threshold (penalizing false positives 2x heavier than false negatives)...")
print("-" * 65)
print(f"  {'Threshold':<12} {'Macro F_0.5':<15} {'Total Matches':<15} {'Notes'}")
print("-" * 65)

for thresh in thresholds:
    val_pred_dict = {s1: set() for s1 in s1_val}
    for s1, cand, prob in zip(s1_val_pairs, cand_val_pairs, val_probs):
        if prob >= thresh:
            val_pred_dict[s1].add(cand)

    macro_f05 = calculate_macro_f05(val_gt_dict, val_pred_dict)
    n_predicted = sum(len(v) for v in val_pred_dict.values())
    threshold_results.append((thresh, macro_f05, n_predicted))

    is_best = macro_f05 > best_f05
    if is_best:
        best_f05 = macro_f05
        best_threshold = thresh

    if abs(thresh - round(thresh, 1)) < 1e-4 or is_best:
        note = "★ (Optimal)" if is_best else ""
        print(f"  {thresh:<12.2f} {macro_f05:<15.4f} {n_predicted:<15,d} {note}")

print("-" * 65)
print(f"[+] Optimal Precision-Heavy Threshold: {best_threshold:.2f} (Macro F_0.5 = {best_f05 * 100:.2f}%)")


NameError: name 'model' is not defined

In [ ]:
# ==============================================================================
# Cell 6 (Corrected): Run Trained Model Inference & Populate matching_results.tsv
# ==============================================================================
import numpy as np
import pandas as pd
from pathlib import Path
from src.features import extract_pairwise_features, FEATURE_NAMES
from src.data_loader import resolve_data_paths

print("[*] Loading test data and preparing feature inference...")
paths = resolve_data_paths()
output_dir = workspace_root / "output"
matching_file = output_dir / "matching_results.tsv"
candidate_file = output_dir / "candidate_pairs.tsv"

# 1. Load test entities and blocking candidates 
# (Ensure your blocking/candidate generation script has populated candidate_pairs.tsv with actual candidate pairs)
if not candidate_file.exists():
    raise FileNotFoundError(f"{candidate_file} not found! Run candidate generation first.")

df_test_candidates = pd.read_csv(candidate_file, sep="\t")
print(f"[+] Loaded {len(df_test_candidates):,} test candidate pairs for inference.")

# 2. Extract or load test features 
# (If features are already extracted, load them; otherwise, compute using your Phase 4 feature extractor)
test_feat_path = output_dir / "test_features.csv"
if test_feat_path.exists():
    df_test_feat = pd.read_csv(test_feat_path)
else:
    print("[*] Extracting features for test candidate pairs...")
    # Assumes your pipeline has an extraction function for test pairs
    df_test_feat = df_test_candidates # Update this if your feature extractor needs to be called explicitly
    
X_test = df_test_feat[FEATURE_NAMES].values

# 3. Predict probabilities using our trained LightGBM model
print("[*] Running model.predict_proba() on test candidate pairs...")
test_probs = model.predict_proba(X_test)[:, 1]
df_test_feat["match_prob"] = test_probs

# 4. Apply optimal decision threshold (best_threshold from Cell 5, e.g., 0.70)
print(f"[*] Applying decision threshold τ* = {best_threshold:.2f}...")
matches_dict = {}
for _, row in df_test_feat.iterrows():
    s1 = row["source1_entity_id"]
    cand = row["candidate_entity_id"]
    prob = row["match_prob"]
    
    if prob >= best_threshold:
        if s1 not in matches_dict:
            matches_dict[s1] = []
        matches_dict[s1].append(str(cand))

# 5. Get all required test Source 1 entities to ensure complete 1.73M coverage
test_s1_file = paths["test"]["source1"]
with open(test_s1_file, "r", encoding="utf-8") as f:
    header = f.readline()
    all_test_s1 = [line.split("\t", 1)[0].strip() for line in f if line.strip()]

# 6. Write final populated matching_results.tsv
print(f"[*] Exporting populated {matching_file.name}...")
with open(matching_file, "w", encoding="utf-8", newline="") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for s1 in all_test_s1:
        matched_list = matches_dict.get(s1, [])
        matched_str = ",".join(matched_list)
        f.write(f"{s1}\t{matched_str}\n")

print(f"[+] Successfully generated real predictions! File size: {matching_file.stat().st_size / 1024 / 1024:.1f} MB")

In [7]:
# ==============================================================================
# Cell 7: Official Validation & Submission Packaging (GenX_H4CK3RS!_submission.zip)
# ==============================================================================
from package_submission import create_submission_zip, validate_against_official_tool

# 1. Package submission zip
print("[*] Packaging submission archive...")
zip_path = create_submission_zip(workspace_root)

# 2. Run official competition submission validator
print("\n[*] Running official competition submission validator...")
validate_against_official_tool(workspace_root)
